# Phase 2: EEG Signal Quality Analysis and Visualization

This notebook executes the Phase 2 EEG signal-quality analysis and visualization pipeline (`imagined_speech.data.signal_quality`) on Google Colab (or locally), building directly on the verified Phase 1 dataset audit.

### Scientific Safeguards
- **No Preprocessing Applied**: Raw EDF recordings are read in-memory in their original physical units without bandpass/notch filtering, resampling, ICA, baseline correction, or normalization.
- **No Silent Deletion**: All discovered recordings and channels (including the 14 candidate EEG channels, auxiliary/metadata channels, and review channels such as `TIME_STAMP_s` and `TIME_STAMP_ms`) are preserved and reported.
- **Candidate EEG vs. Auxiliary/Review Separation**: Channel-quality screening explicitly distinguishes `candidate_eeg` channels from `non_eeg_metadata_or_aux`, `non_eeg_counter_signal`, and `unknown_needs_review` channels.
- **Unverified Participant Prefixes**: Filename prefixes remain strictly labeled as `unverified_candidate_prefix` and are not assumed to represent verified human participant IDs.
- **Measured vs. Heuristic Separation**: Direct measurements (`measured_*`) are reported separately from configurable heuristic flags (`heuristic_*`).

## 1. Mount Google Drive

In [ ]:
import os
import sys
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Google Colab: {IN_COLAB}')

## 2. Clone or Update the Repository

In [ ]:
import subprocess

REPO_URL = 'https://github.com/ACT2039/imagined-speech-eeg.git'
COLAB_REPO_DIR = Path('/content/imagined-speech-eeg')

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        print(f'Cloning {REPO_URL} into {COLAB_REPO_DIR}...')
        subprocess.run(['git', 'clone', REPO_URL, str(COLAB_REPO_DIR)], check=True)
    elif (COLAB_REPO_DIR / '.git').is_dir():
        print(f'Updating existing repository at {COLAB_REPO_DIR}...')
        subprocess.run(['git', '-C', str(COLAB_REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        raise RuntimeError(
            f'Directory {COLAB_REPO_DIR} exists but is not a valid Git repository.'
        )
    os.chdir(COLAB_REPO_DIR)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError(
        f'pyproject.toml not found in working directory {REPO_ROOT}. '
        'Ensure the notebook is running from the repository root.'
    )

print(f'Repository root: {REPO_ROOT}')

## 3. Install the Package

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'], check=True)

src_dir = str((REPO_ROOT / 'src').resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

import imagined_speech
print(f'Installed imagined_speech version: {imagined_speech.__version__}')

## 4. Verify Existing Phase 1 Audit Reports
Confirm that `manifest.csv`, `channel_audit.csv`, and `audit_summary.json` produced by `notebooks/01_phase1_dataset_audit.ipynb` exist and are non-empty before starting Phase 2 analysis.

In [ ]:
import json
import pandas as pd
from IPython.display import display
from imagined_speech.data.signal_quality import load_and_validate_manifest

PHASE1_AUDIT_DIR = Path(
    os.environ.get(
        'ISR_PHASE1_AUDIT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase1_audit'
        if IN_COLAB
        else './outputs/phase1_audit',
    )
)
PHASE2_OUTPUT_DIR = Path(
    os.environ.get(
        'ISR_OUTPUT_DIR',
        '/content/drive/MyDrive/imagined_speech_outputs/phase2_signal_quality'
        if IN_COLAB
        else './outputs/phase2_signal_quality',
    )
)
CONFIG_PATH = REPO_ROOT / 'configs' / 'default_config.yaml'

REQUIRED_PHASE1_FILES = ('manifest.csv', 'channel_audit.csv', 'audit_summary.json')
missing_phase1 = [
    name
    for name in REQUIRED_PHASE1_FILES
    if not (PHASE1_AUDIT_DIR / name).is_file() or (PHASE1_AUDIT_DIR / name).stat().st_size == 0
]
if missing_phase1:
    raise FileNotFoundError(
        f'Required Phase 1 audit reports are missing or empty in {PHASE1_AUDIT_DIR}: '
        f'{missing_phase1}. Please run notebooks/01_phase1_dataset_audit.ipynb first.'
    )

manifest_path = PHASE1_AUDIT_DIR / 'manifest.csv'
channel_audit_path = PHASE1_AUDIT_DIR / 'channel_audit.csv'
audit_summary_path = PHASE1_AUDIT_DIR / 'audit_summary.json'

manifest_df = load_and_validate_manifest(manifest_path)
channel_audit_df = pd.read_csv(channel_audit_path)
phase1_summary = json.loads(audit_summary_path.read_text(encoding='utf-8'))

print(f'Phase 1 audit directory verified: {PHASE1_AUDIT_DIR}')
print(f'  - Discovered / Valid EDF files: {phase1_summary.get("n_discovered_files")} / {phase1_summary.get("n_valid_edf_files")}')
print(f'  - Sampling rates (Hz):          {phase1_summary.get("distinct_sampling_rates_hz")}')
print(f'  - Duration stats (sec):         {phase1_summary.get("duration_sec_stats")}')
print(f'  - Channel roles discovered:     {phase1_summary.get("channel_roles_discovered")}')
display(manifest_df.head(5))

## 5. Verify or Extract Dataset Archive and Resolve `DATA_ROOT`
Use the confirmed extracted dataset root (`/content/data/raw/Imagined_speech_EEG_edf`) when already present in the runtime, or extract `/content/drive/MyDrive/datasets/Imagined_speech_EEG_edf.zip` into `/content/data/raw` in a fresh Colab runtime and locate `Char/`, `Digit/`, and `Image/`.

In [ ]:
import zipfile
from imagined_speech.exceptions import DatasetDiscoveryError

DRIVE_ZIP_PATH = Path(
    os.environ.get(
        'ISR_DRIVE_ZIP',
        '/content/drive/MyDrive/datasets/Imagined_speech_EEG_edf.zip',
    )
)
EXTRACT_ROOT = Path(
    os.environ.get(
        'ISR_EXTRACT_ROOT',
        '/content/data/raw' if IN_COLAB else './data/raw',
    )
)
CONFIRMED_COLAB_DATA_ROOT = Path('/content/data/raw/Imagined_speech_EEG_edf')
EXPECTED_TASKS = ('Char', 'Digit', 'Image')


def _has_expected_task_dirs(candidate: Path, expected_tasks: tuple[str, ...] = EXPECTED_TASKS) -> tuple[bool, list[str]]:
    if not candidate.is_dir():
        return False, []
    child_dirs = {
        child.name.lower(): child.name
        for child in sorted(candidate.iterdir(), key=lambda p: p.name.lower())
        if child.is_dir()
    }
    expected_lower = {t.lower() for t in expected_tasks}
    if expected_lower.issubset(child_dirs.keys()):
        return True, [child_dirs[t.lower()] for t in expected_tasks]
    return False, []


def locate_dataset_root(
    search_root: Path,
    expected_tasks: tuple[str, ...] = EXPECTED_TASKS,
) -> tuple[Path, list[str]]:
    """Locate the directory containing Char/, Digit/, and Image/ subdirectories."""
    if not search_root.exists() or not search_root.is_dir():
        raise DatasetDiscoveryError(
            f'Extraction directory does not exist or is not a directory: {search_root}'
        )

    candidate_dirs = [search_root] + sorted(
        (
            d
            for d in search_root.rglob('*')
            if d.is_dir() and not any(part.startswith('.') or part == '__MACOSX' for part in d.parts)
        ),
        key=lambda p: (len(p.parts), p.as_posix().lower()),
    )
    for cand in candidate_dirs:
        matched, folders = _has_expected_task_dirs(cand, expected_tasks)
        if matched:
            return cand.resolve(), folders

    observed_dirs = [p.relative_to(search_root).as_posix() for p in candidate_dirs[1:15]]
    raise DatasetDiscoveryError(
        f'Could not find a directory under {search_root} containing required '
        f'category folders {list(expected_tasks)}. Observed subdirectories: {observed_dirs}'
    )


existing_edfs = (
    [p for p in EXTRACT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() == '.edf']
    if EXTRACT_ROOT.exists()
    else []
)
if existing_edfs:
    print(f'Skipping ZIP extraction: found {len(existing_edfs)} existing .edf files under {EXTRACT_ROOT}.')
else:
    if not DRIVE_ZIP_PATH.exists():
        raise FileNotFoundError(
            f'Dataset ZIP archive not found at {DRIVE_ZIP_PATH}. '
            'Verify that Google Drive is mounted at /content/drive and that '
            'MyDrive/datasets/Imagined_speech_EEG_edf.zip exists.'
        )
    if not DRIVE_ZIP_PATH.is_file() or not zipfile.is_zipfile(DRIVE_ZIP_PATH):
        raise ValueError(f'Path {DRIVE_ZIP_PATH} exists but is not a valid ZIP archive.')

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'Extracting {DRIVE_ZIP_PATH} -> {EXTRACT_ROOT} ...')
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_ROOT)

env_data_root = os.environ.get('ISR_DATA_ROOT')
if env_data_root:
    DATA_ROOT, discovered_categories = locate_dataset_root(Path(env_data_root))
elif IN_COLAB and _has_expected_task_dirs(CONFIRMED_COLAB_DATA_ROOT)[0]:
    DATA_ROOT = CONFIRMED_COLAB_DATA_ROOT.resolve()
    discovered_categories = _has_expected_task_dirs(CONFIRMED_COLAB_DATA_ROOT)[1]
else:
    DATA_ROOT, discovered_categories = locate_dataset_root(EXTRACT_ROOT)

print(f'Resolved DATA_ROOT:         {DATA_ROOT}')
print(f'Discovered category folders: {discovered_categories}')
print(f'PHASE2_OUTPUT_DIR:          {PHASE2_OUTPUT_DIR}')

## 6. Load Configuration and Deterministically Select Representative Recordings
Reuse the project's default configuration (`configs/default_config.yaml`) and deterministically select representative recordings across `Char`, `Digit`, and `Image`.

In [ ]:
from dataclasses import asdict
from imagined_speech.config import load_config
from imagined_speech.data.signal_quality import select_representative_recordings
from imagined_speech.utils import set_global_seed, setup_logging

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Configuration file not found at {CONFIG_PATH}')

config = load_config(
    config_path=CONFIG_PATH,
    data_root=DATA_ROOT,
    output_dir=PHASE2_OUTPUT_DIR,
)
setup_logging(config.log_level)
set_global_seed(config.seed)

selected_inputs = select_representative_recordings(
    manifest_df=manifest_df,
    data_root=config.data_root,
    expected_tasks=config.discovery.expected_tasks,
    recordings_per_task=config.signal_quality.recordings_per_task,
    specific_recordings=config.signal_quality.specific_recordings,
    seed=config.seed,
)
print(f'Selected {len(selected_inputs)} representative recordings (recordings_per_task={config.signal_quality.recordings_per_task}, seed={config.seed}):')
display(pd.DataFrame([asdict(s) for s in selected_inputs]))

## 7. Run the Phase 2 EEG Signal Quality Analysis
Inspect EDF headers and raw signals in original physical units, compute per-channel statistics and heuristic flags using configured thresholds, and save reports and figures to `PHASE2_OUTPUT_DIR`.

In [ ]:
from imagined_speech.data.signal_quality import run_signal_quality_analysis

sq_result = run_signal_quality_analysis(
    config=config,
    manifest_source=manifest_df,
    output_dir=PHASE2_OUTPUT_DIR,
    auto_run_audit_if_missing=False,
    save_outputs=True,
)
print('=== Phase 2 Signal Quality Summary ===')
print(json.dumps(sq_result.summary, indent=2))

## 8. Review Channel-Quality Screening (Candidate EEG vs. Auxiliary/Metadata/Review Channels)

In [ ]:
stats_df = pd.read_csv(PHASE2_OUTPUT_DIR / 'signal_statistics.csv')

print(f'Total channel statistics rows in signal_statistics.csv: {len(stats_df)}')
print('\nChannel counts by audited role and candidate EEG status:')
display(
    stats_df.groupby(['is_candidate_eeg', 'audited_channel_role', 'raw_channel_name'])
    .size()
    .reset_index(name='n_inspected_recordings')
)

eeg_stats_df = stats_df[stats_df['is_candidate_eeg'].astype(bool)]
non_eeg_stats_df = stats_df[~stats_df['is_candidate_eeg'].astype(bool)]

print(f'\nCandidate EEG channels preview ({len(eeg_stats_df)} rows across {eeg_stats_df["raw_channel_name"].nunique()} channels):')
display(
    eeg_stats_df[
        [
            'relative_source_path',
            'task_category',
            'raw_channel_name',
            'physical_unit',
            'measured_min',
            'measured_max',
            'measured_mean',
            'measured_std',
            'measured_amplitude_range_ptp',
            'heuristic_quality_flags',
        ]
    ].head(14)
)

print(f'\nAuxiliary / Metadata / Unknown review channels preview ({len(non_eeg_stats_df)} rows, including TIME_STAMP_s / TIME_STAMP_ms if present):')
display(
    non_eeg_stats_df[
        [
            'relative_source_path',
            'task_category',
            'raw_channel_name',
            'audited_channel_role',
            'physical_unit',
            'measured_min',
            'measured_max',
            'measured_std',
            'heuristic_quality_flags',
        ]
    ].head(20)
)

## 9. Display Signal Quality Report and Generated Figures

In [ ]:
from IPython.display import Image, Markdown

report_path = PHASE2_OUTPUT_DIR / 'signal_quality_report.md'
display(Markdown(report_path.read_text(encoding='utf-8')))

for fig_path in sq_result.generated_figures:
    print(f'Figure: {fig_path}')
    display(Image(filename=fig_path))

## 10. Verify Persistent Phase 2 Outputs in Google Drive
Confirm that `signal_statistics.csv`, `signal_quality_summary.json`, `signal_quality_report.md`, and `figures/` exist and are non-empty in `/content/drive/MyDrive/imagined_speech_outputs/phase2_signal_quality/`.

In [ ]:
REQUIRED_PHASE2_FILES = (
    'signal_statistics.csv',
    'signal_quality_summary.json',
    'signal_quality_report.md',
)

missing_or_empty: list[str] = []
verification_rows: list[dict[str, object]] = []

for filename in REQUIRED_PHASE2_FILES:
    artifact_path = PHASE2_OUTPUT_DIR / filename
    exists = artifact_path.is_file()
    size_bytes = artifact_path.stat().st_size if exists else 0
    if not exists or size_bytes == 0:
        missing_or_empty.append(f'{artifact_path} (exists={exists}, size_bytes={size_bytes})')
    verification_rows.append(
        {
            'artifact': filename,
            'path': str(artifact_path),
            'exists': exists,
            'size_bytes': size_bytes,
        }
    )

figures_dir = PHASE2_OUTPUT_DIR / 'figures'
png_files = sorted(p for p in figures_dir.glob('*.png') if p.is_file() and p.stat().st_size > 0) if figures_dir.is_dir() else []
if not png_files:
    missing_or_empty.append(f'{figures_dir} (no non-empty .png figures found)')

if missing_or_empty:
    raise RuntimeError(
        'Phase 2 output verification failed. Missing or empty artifacts: '
        + ', '.join(missing_or_empty)
    )

print(f'Verified {len(REQUIRED_PHASE2_FILES)} report files and {len(png_files)} PNG figure(s) in {PHASE2_OUTPUT_DIR}:')
display(pd.DataFrame(verification_rows))
for fig_file in png_files:
    print(f'  - figures/{fig_file.name} ({fig_file.stat().st_size} bytes)')